# 03 - Seasonality, crashes and volatility (Q3, Q5)

Q3: which months and crops crash predictably?  
Q5: how does volatility differ between the perishable (tomato) and storables (onion, potato)?

Crash (spec 9.2): within the next 14 days the price falls below 70% of its trailing 30-day median, computed only from days with data.

In [1]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import numpy as np
import pandas as pd
from mandipulse import viz
from mandipulse.viz import COMMODITIES, COMMODITY_COLORS, INK, INK_2, MUTED, load
pd.set_option("display.width", 160)
viz.style()

## Q3 - seasonal price pattern and crash rate by month

In [2]:
s = load("phase3", "q3_seasonality_by_month")
fig, ax = viz.figure("Q3. Seasonal price pattern by crop",
                     "Price index = month median / annual median (1.0 = a typical month), median across years and states")
for c in COMMODITIES:
    g = s[s["commodity"] == c]
    ax.plot(g["month"], g["median_price_index"], color=COMMODITY_COLORS[c], marker="o", markersize=5, label=c)
viz.label_line_ends(ax, s, "month", "median_price_index", "commodity", fmt="{:.2f}")
ax.axhline(1, color=MUTED, linewidth=0.8)
ax.set_xticks(range(1, 13), ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"])
ax.set_ylabel("price index (x annual median)")
viz.legend(ax)
viz.save(fig, "q3_price_index_by_month")
s.pivot(index="month", columns="commodity", values="median_price_index")

commodity,Onion,Potato,Tomato
month,,,
1,1.314,0.831,0.751
2,1.350,0.756,0.666
3,0.930,0.755,0.685
4,0.731,0.919,0.705
5,0.711,1.000,0.814
6,0.846,1.070,1.332
7,1.002,1.068,1.785
8,1.046,1.085,1.544
9,0.997,1.127,1.141


In [3]:
grid = s.pivot(index="commodity", columns="month", values="crash_rate_pct").loc[COMMODITIES]
fig, ax = viz.figure("Q3. How often prices crash, by month",
                     "% of market-days followed within 14 days by a fall below 70% of the 30-day median", height=3.8)
cmap = LinearSegmentedColormap.from_list("seq", viz.SEQUENTIAL)
im = ax.imshow(grid.values, cmap=cmap, aspect="auto", vmin=0)
ax.grid(False)
ax.set_xticks(range(12), ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"])
ax.set_yticks(range(len(grid.index)), grid.index)
vmax = np.nanmax(grid.values)
for i in range(grid.shape[0]):
    for j in range(grid.shape[1]):
        v = grid.values[i, j]
        ax.text(j, i, f"{v:.1f}", ha="center", va="center", fontsize=8.5,
                color="white" if v > 0.55 * vmax else INK)
fig.colorbar(im, ax=ax, label="crash rate, %", shrink=0.8)
viz.save(fig, "q3_crash_rate_heatmap")
grid

month,1,2,3,4,5,6,7,8,9,10,11,12
commodity,,,,,,,,,,,,
Tomato,41.49,21.52,14.55,13.66,15.99,9.20,15.64,39.98,33.75,10.25,33.41,64.09
Onion,20.62,22.61,35.99,25.11,8.90,2.58,3.20,3.79,5.43,3.88,18.00,42.05
Potato,22.16,8.32,3.33,1.92,2.14,1.63,1.98,2.39,1.93,2.00,11.85,38.55


In [4]:
display(load("phase3", "q3_crash_rate_by_state_month").pivot_table(
    index=["commodity", "state"], columns="month", values="crash_rate_pct"))
load("phase3", "q3_crash_lead")

month                        1      2      3      4      5      6      7      8      9      10     11     12
commodity state                                                                                             
Onion     Gujarat         26.44  33.23  50.05  29.90   8.84   2.89   4.68   4.02   9.14   4.53  35.25  53.33
          Madhya Pradesh  31.32  36.07  47.49  36.50  30.29  11.42  14.83  17.87  21.39  13.53  35.69  47.26
          Maharashtra     38.90  37.72  47.85  17.61   5.29   1.92   3.78   3.20   5.03   6.92  35.97  53.81
          Uttar Pradesh   13.87  15.22  29.51  24.43   6.16   1.03   0.57   1.24   2.38   1.31   8.86  36.45
Potato    Gujarat         39.72  19.13   8.67   3.06   4.08   1.79   1.09   2.60   2.53   2.18  12.16  31.46
          Madhya Pradesh  23.62  18.73  11.68   9.80  10.45   6.73  10.98  12.37   8.54   9.70  20.82  31.35
          Maharashtra     22.65  16.75   8.09   3.50   4.46   3.74   3.57   1.76   1.59   2.90   8.68  32.11
          Uttar Pradesh   20.60   5.10   1.15   0.78   0.83   0.76   0.65   1.07   1.07   0.82  10.87  40.81
Tomato    Gujarat         57.24  37.72  32.21  17.25   9.20  13.82  23.51  65.79  46.31  21.01  45.50  67.68
          Madhya Pradesh  44.35  30.69  19.24  18.47   7.96  11.97  20.95  43.40  41.75  16.58  38.00  58.37
          Maharashtra     52.05  41.97  34.37  26.35  12.25  26.07  37.36  67.30  39.88  24.63  53.54  59.71
          Uttar Pradesh   37.17  13.92   7.48  10.40  18.45   4.72   9.01  29.98  29.73   4.95  27.01  65.57

,commodity,n_crash_days,median_price_vs_trailing_median_pct_crash_days,median_price_vs_trailing_median_pct_other_days
0,Onion,91830,82.0,100.0
1,Potato,45325,82.1,100.0
2,Tomato,146268,79.5,100.6


## Q5 - volatility: perishable vs storable

In [5]:
v = load("phase3", "q5_volatility_by_commodity").set_index("commodity").loc[COMMODITIES].reset_index()
viz.style()
fig, axes = plt.subplots(1, 2, figsize=(10, 4.6))
viz.header(fig, "Q5. Price volatility: tomato vs onion and potato",
           "Median over market-years with >= 60 report days (main period)")
panels = [("median_cv", "coefficient of variation (std / mean)", "{:.2f}"),
          ("median_avg_abs_daily_change_pct", "average absolute day-to-day change, %", "{:.1f}%")]
for ax, (col, label, fmt) in zip(axes, panels, strict=True):
    bars = ax.bar(v["commodity"], v[col], color=[COMMODITY_COLORS[c] for c in v["commodity"]], width=0.55)
    for b, val in zip(bars, v[col], strict=True):
        ax.annotate(fmt.format(val), (b.get_x() + b.get_width() / 2, val), xytext=(0, 3),
                    textcoords="offset points", ha="center", fontsize=9, color=INK_2)
    ax.set_title(label, loc="left", fontsize=10, color=INK_2)
    ax.set_ylim(0)
viz.save(fig, "q5_volatility_by_commodity")
v

,commodity,category,n_market_years,median_cv,median_avg_abs_daily_change_pct,median_max_drawdown_pct
0,Tomato,perishable,2322,0.491,5.6,74.1
1,Onion,storable,2640,0.394,4.7,66.9
2,Potato,storable,2253,0.249,2.7,52.5


In [6]:
vy = load("phase3", "q5_volatility_by_year")
fig, ax = viz.figure("Q5. Volatility every year", "Median coefficient of variation of daily prices within a market-year")
for c in COMMODITIES:
    g = vy[vy["commodity"] == c]
    ax.plot(g["year"], g["median_cv"], color=COMMODITY_COLORS[c], marker="o", markersize=5, label=c)
viz.label_line_ends(ax, vy, "year", "median_cv", "commodity", fmt="{:.2f}")
ax.set_ylabel("median CV")
ax.set_ylim(0)
viz.legend(ax)
viz.save(fig, "q5_cv_by_year")
vy.pivot(index="year", columns="commodity", values="median_cv")

commodity,Onion,Potato,Tomato
year,,,
2018,0.428,0.310,0.443
2019,0.698,0.250,0.388
2020,0.552,0.336,0.491
2021,0.282,0.214,0.581
2022,0.254,0.254,0.404
2023,0.466,0.194,0.950
2024,0.379,0.313,0.463
2025,0.298,0.127,0.603
